# Finish the Grover CSP study on Kaggle

Before running: **Add Input → Upload** `grovers_colab.zip` and the latest `study_threshold.csv` from Google Drive (`grovers_colab/results/`).
Settings (right panel): **Internet on**, Accelerator **None**.

Best way to run: **Save Version → Save & Run All (Commit)**. It runs in the background for up to 12 hours, even with the browser closed. The results appear in the version's **Output** tab.

In [ ]:
import os, shutil, zipfile

INPUT = '/kaggle/input'
WORK = '/kaggle/working/grovers'

def find(name):
    return [os.path.join(d, name) for d, _, fs in os.walk(INPUT, followlinks=True) if name in fs]

print('Files in', INPUT)
for d, _, fs in os.walk(INPUT, followlinks=True):
    for f in fs:
        print('  ', os.path.join(d, f))

os.makedirs(WORK, exist_ok=True)
if find('study.py'):
    src = os.path.dirname(find('study.py')[0])
    shutil.copytree(src, WORK, dirs_exist_ok=True)
elif find('grovers_colab.zip'):
    src = find('grovers_colab.zip')[0]
    zipfile.ZipFile(src).extractall(WORK)  # Kaggle left the zip packed
else:
    raise SystemExit('No input found: click "Add Input" in the right panel and attach the dataset with grovers_colab.zip')
print('code taken from', src)
os.makedirs(f'{WORK}/results', exist_ok=True)
os.makedirs(f'{WORK}/figures', exist_ok=True)

# Use whichever study_threshold.csv has the most finished simulations
def rows(p):
    return sum(1 for _ in open(p, encoding='utf-8'))
local = f'{WORK}/results/study_threshold.csv'
cands = [c for c in find('study_threshold.csv') + [local] if os.path.exists(c)]
for c in cands:
    print(f'{rows(c) - 1:4d} rows  {c}')
best = max(cands, key=rows)
if best != local:
    shutil.copy(best, local)
print('resuming from', best)
os.chdir(WORK)

In [ ]:
# Same versions as the paper's Methods section
!pip install -q qiskit==2.2.3 qiskit-aer==0.17.2 qiskit-ibm-runtime==0.45.1

In [ ]:
# Several hours. Each finished simulation is appended to results/ immediately.
!python study.py all 2>&1 | tee -a results/study_kaggle_run.log

In [ ]:
# One file to download from the Output tab
os.chdir('/kaggle/working')
shutil.make_archive('/kaggle/working/study_results', 'zip', WORK, '.')
print(os.path.getsize('/kaggle/working/study_results.zip') // 1024, 'KB -> study_results.zip')